# 04 – Intent và mock users


In [1]:
from __future__ import annotations

import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

print("PROJECT_ROOT =", PROJECT_ROOT)
assert (PROJECT_ROOT / "app").exists(), "Chạy notebook từ vaxcare-ai hoặc notebooks/"

PROJECT_ROOT = d:\TTTN-VaxCare-2026\vaxcare-ai


## 1. Nạp danh sách vaccine từ DB (đã export)

In [2]:
vaccines_path = PROJECT_ROOT / "data" / "mock" / "vaccines_from_db.json"
with open(vaccines_path, encoding="utf-8") as f:
    vaccines = json.load(f)

print(f"Số vaccine: {len(vaccines)}")
for v in vaccines:
    print(f"  [{v['vaccine_id']:2d}] {v['short_name']:20s} | {v['target_disease']}")

Số vaccine: 13
  [ 1] BCG                  | Bệnh lao
  [ 2] Viêm gan B           | Viêm gan B
  [ 3] DTaP/Tdap            | Bạch hầu, ho gà, uốn ván
  [ 4] IPV                  | Bệnh bại liệt
  [ 5] Hib                  | Các bệnh xâm lấn do Hib
  [ 6] MMR                  | Sởi, quai bị, rubella
  [ 7] Thủy đậu             | Thủy đậu
  [ 8] Phế cầu              | Bệnh do phế cầu
  [ 9] Viêm não Nhật Bản    | Viêm não Nhật Bản
  [10] Cúm                  | Cúm mùa
  [11] HPV                  | Ung thư cổ tử cung, sùi mào gà và các bệnh liên quan HPV
  [12] COVID-19             | COVID-19 (SARS-CoV-2)
  [13] Zona / Shingrix      | Zona (Herpes Zoster)


## 2. Seed câu hỏi theo 6 intent + nhân rộng

6 nhóm (theo `app/chatbot/intent/labels.py`):
- `faq_vaccine`
- `schedule_advice`
- `history_lookup`
- `vaxcare_lookup`
- `chitchat`
- `out_of_scope`

In [3]:
SEED: dict[str, list[str]] = {
    "faq_vaccine": [
        "Vaccine {v} phòng bệnh gì?",
        "{v} dùng để làm gì?",
        "Công dụng của vắc xin {v} là gì?",
        "Ai nên tiêm {v}?",
        "Đối tượng sử dụng vaccine {v}?",
        "Có chống chỉ định gì khi tiêm {v}?",
        "Lưu ý sau khi tiêm {v}?",
        "Tác dụng không mong muốn của {v}?",
        "{v} tiêm mấy mũi?",
        "Khoảng cách giữa các mũi {v}?",
        "Vaccine phòng {d} tên gì?",
        "Thông tin cần biết trước khi tiêm {v}",
        "{v} có an toàn không?",
        "Phác đồ tiêm {v} như thế nào?",
        "{v} khác gì so với các vaccine khác phòng {d}?",
    ],
    "schedule_advice": [
        "Tôi {age} tuổi, nên kiểm tra những vaccine nào?",
        "Con tôi {age} tháng, mốc tiêm nào quan trọng?",
        "Lộ trình tiêm chủng cho người {age} tuổi",
        "Ở độ tuổi {age} nên tiêm gì?",
        "Tôi sinh năm {year}, còn thiếu mũi nào?",
        "Trẻ {age} tháng tiêm những mũi nào theo TCMR?",
        "Người lớn {age} tuổi cần quan tâm vaccine gì?",
        "Mốc tiêm tiếp theo sau {age} tháng tuổi",
        "Lịch tiêm tham khảo cho {age} tuổi",
        "Tôi {age} tuổi đã tiêm {v} mũi 1, tiếp theo?",
        "Bao nhiêu tuổi thì tiêm {v}?",
        "Khi nào nên tiêm {v}?",
        "Lịch tiêm {v} theo tuổi",
        "Trẻ sơ sinh nên tiêm những gì?",
        "Người trên 50 tuổi nên tiêm vaccine nào?",
    ],
    "history_lookup": [
        "Tôi đã tiêm mũi 1 rồi, tiếp theo cần làm gì?",
        "Xem lịch sử tiêm của tôi",
        "Tôi đã tiêm những mũi nào?",
        "Hồ sơ tiêm của tôi còn thiếu gì?",
        "Đối chiếu lịch sử tiêm với phác đồ",
        "Tôi tiêm {v} mũi 1 ngày {date}, khi nào mũi 2?",
        "Kiểm tra xem tôi đã đủ mũi {v} chưa",
        "Lịch sử tiêm chủng trong tài khoản",
        "Tôi quên đã tiêm bao nhiêu mũi {v}",
        "Mũi gần nhất tôi tiêm là gì?",
        "Tôi đã tiêm đủ chưa?",
        "So sánh hồ sơ tiêm với lịch khuyến cáo",
        "Tôi tiêm trễ mũi {v} thì sao?",
        "Còn thiếu mũi nào theo hồ sơ?",
        "Giải thích dữ liệu tiêm đã ghi nhận",
    ],
    "vaxcare_lookup": [
        "Tìm cơ sở tiêm {v} gần đây",
        "Đặt lịch tiêm {v}",
        "Có phòng khám nào tiêm {v} không?",
        "Giá vaccine {v} khoảng bao nhiêu?",
        "Dẫn tôi đến trang đặt lịch",
        "Tra cứu cơ sở VaxCare",
        "Vaccine {v} còn hàng không?",
        "Đăng ký tiêm tại VaxCare",
        "Tìm vaccine {v} trên hệ thống",
        "Cơ sở nào tiêm {v} ở quận 1?",
        "Mở trang chi tiết vaccine {v}",
        "Tôi muốn đặt lịch tiêm chủng",
        "Danh sách cơ sở tiêm chủng",
        "Hướng dẫn đặt lịch tiêm",
        "Tìm địa điểm tiêm gần nhất",
    ],
    "chitchat": [
        "Xin chào",
        "Hello",
        "Chào bot",
        "Bạn có thể giúp gì?",
        "Cảm ơn",
        "Thanks",
        "Tạm biệt",
        "Bạn là ai?",
        "Hi VaxCare",
        "Chào buổi sáng",
        "Ok cảm ơn bạn",
        "Được rồi",
        "Hẹn gặp lại",
        "Bạn khỏe không?",
        "Cho mình hỏi chút",
    ],
    "out_of_scope": [
        "Tôi bị đau ngực khó thở phải làm sao?",
        "Chẩn đoán giúp tôi bị bệnh gì",
        "Uống thuốc gì để hết sốt sau tiêm?",
        "Tôi bị dị ứng nặng sau tiêm, cấp cứu thế nào?",
        "Kê đơn thuốc giúp tôi",
        "Tôi có nên uống kháng sinh không?",
        "Triệu chứng này có phải covid không?",
        "Mổ ruột thừa xong bao lâu được tiêm?",
        "Công thức pha vaccine tại nhà",
        "Giá cổ phiếu công ty vaccine",
        "Hôm nay thời tiết thế nào?",
        "Kể chuyện cười đi",
        "Làm sao để tự tiêm tại nhà?",
        "Tôi muốn mua vaccine về tự tiêm",
        "Chữa bệnh zona bằng cách nào không cần tiêm?",
    ],
}

AGES = ["0", "2", "4", "6", "9", "12", "15", "18", "24", "36", "5", "10", "14", "18", "20", "25", "30", "40", "50", "60"]
YEARS = ["2024", "2020", "2010", "2006", "1996", "1970"]
DATES = ["01/06/2026", "15/03/2026", "20/08/2025"]

random.seed(42)

def fill_template(tmpl: str, v: dict) -> str:
    return (
        tmpl.replace("{v}", v["short_name"])
        .replace("{d}", v["target_disease"].split(",")[0].strip())
        .replace("{age}", random.choice(AGES))
        .replace("{year}", random.choice(YEARS))
        .replace("{date}", random.choice(DATES))
    )

rows: list[dict] = []
for label, templates in SEED.items():
    # Giữ nguyên seed không placeholder
    for t in templates:
        if "{" not in t:
            rows.append({"text": t, "label": label, "source": "seed"})

    # Nhân rộng với vaccine (chỉ các template có {v} hoặc {d})
    need_v = [t for t in templates if "{v}" in t or "{d}" in t]
    for v in vaccines:
        for t in need_v:
            rows.append({"text": fill_template(t, v), "label": label, "source": "expand_vaccine"})

    # Nhân rộng thêm vài biến thể tuổi cho schedule
    if label == "schedule_advice":
        for t in templates:
            if "{age}" in t or "{year}" in t:
                for _ in range(3):
                    rows.append({"text": fill_template(t, random.choice(vaccines)), "label": label, "source": "expand_age"})

df = pd.DataFrame(rows).drop_duplicates(subset=["text"]).reset_index(drop=True)
print("Tổng câu sau nhân rộng (unique):", len(df))
print(df["label"].value_counts())
df.sample(8, random_state=1)

Tổng câu sau nhân rộng (unique): 482
label
faq_vaccine        195
vaxcare_lookup     111
schedule_advice     83
history_lookup      63
chitchat            15
out_of_scope        15
Name: count, dtype: int64


,text,label,source
31,DTaP/Tdap dùng để làm gì?,faq_vaccine,expand_vaccine
246,Bao nhiêu tuổi thì tiêm Zona / Shingrix?,schedule_advice,expand_vaccine
185,Có chống chỉ định gì khi tiêm Zona / Shingrix?,faq_vaccine,expand_vaccine
110,Có chống chỉ định gì khi tiêm Phế cầu?,faq_vaccine,expand_vaccine
90,Vaccine Thủy đậu phòng bệnh gì?,faq_vaccine,expand_vaccine
168,Ai nên tiêm COVID-19?,faq_vaccine,expand_vaccine
256,Lộ trình tiêm chủng cho người 18 tuổi,schedule_advice,expand_age
117,Phế cầu có an toàn không?,faq_vaccine,expand_vaccine


In [4]:
# Lưu seed + expanded (chưa split train/test – ngày 29 sẽ gộp train.csv)
out_dir = PROJECT_ROOT / "intent_data"
out_dir.mkdir(parents=True, exist_ok=True)

seed_path = out_dir / "seed_expanded.csv"
df.to_csv(seed_path, index=False, encoding="utf-8-sig")
print("Saved:", seed_path)
print("Per label counts:")
print(df.groupby("label").size())

Saved: d:\TTTN-VaxCare-2026\vaxcare-ai\intent_data\seed_expanded.csv
Per label counts:
label
chitchat            15
faq_vaccine        195
history_lookup      63
out_of_scope        15
schedule_advice     83
vaxcare_lookup     111
dtype: int64


## 3. Mock users (15 hồ sơ)

In [5]:
mock_path = PROJECT_ROOT / "data" / "mock" / "mock_users.json"
with open(mock_path, encoding="utf-8") as f:
    mock_users = json.load(f)

print(f"Số hồ sơ: {len(mock_users)}")
for u in mock_users:
    n = len(u.get("records") or [])
    age_months = u.get("age_months")
    if age_months is not None:
        age_str = f"{age_months/12:5.1f}y ({age_months}m)"
    else:
        age_str = "  n/a"
    note = u.get("note") or u.get("scenario") or ""
    print(
        f"  id={u.get('user_id', '?'):>3} | "
        f"{str(u.get('full_name', '')):20s} | "
        f"age≈{age_str} | records={n:2d} | {note}"
    )

Số hồ sơ: 15
  id=101 | Nguyen Van A         | age≈  1.2y (14m) | records= 6 | 
  id=102 | Tran Thi B           | age≈  1.5y (18m) | records= 7 | 
  id=103 | Le Van C             | age≈  0.5y (6m) | records= 2 | 
  id=104 | Pham Thi D           | age≈  3.0y (36m) | records= 3 | 
  id=105 | Hoang Van E          | age≈ 20.0y (240m) | records= 1 | 
  id=106 | Vu Thi F             | age≈ 10.0y (120m) | records= 2 | 
  id=107 | Do Van G             | age≈ 50.0y (600m) | records= 0 | 
  id=108 | Bui Thi H            | age≈  0.8y (9m) | records= 6 | 
  id=109 | Ngo Van I            | age≈  0.2y (2m) | records= 0 | 
  id=110 | Dang Thi K           | age≈  2.0y (24m) | records= 7 | 
  id=111 | Mai Van L            | age≈  2.0y (24m) | records= 2 | Trễ mũi: 24 tháng chỉ BCG + HepB1
  id=112 | Tran Thi M           | age≈  0.5y (6m) | records= 1 | Thai phụ giả lập; đã Td mũi 1
  id=113 | Pham Van N           | age≈ 52.0y (624m) | records= 1 | ≥50 tuổi; đã cúm, chưa Zona
  id=114 | Le Thi O        

## 4. Loader quy tắc tuổi

In [6]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from app.chatbot.rules.age_rules_loader import load_age_rules, rules_for_age, expected_doses_by_vaccine

rules = load_age_rules()
print(f"Số rule: {len(rules)}")
display(rules.head(8))

print("\n--- Rules cho 14 tháng tuổi ---")
display(rules_for_age(14))

print("\n--- Expected doses map (14 tháng) ---")
exp = expected_doses_by_vaccine(14)
for vid, doses in exp.items():
    print(vid, "->", [(d["vaccine_short"], d["dose_number"]) for d in doses])

Số rule: 40


,rule_id,age_group,age_from_months,age_to_months,vaccine_id,vaccine_short,dose_number,interval_days_min,source_doc,note
0,1,Sơ sinh,0,1,1,BCG,1,0,tt10-2024,Tiêm trong da càng sớm càng tốt trong vòng 1 t...
1,2,Sơ sinh,0,1,2,Viêm gan B,1,0,tt10-2024,Liều sơ sinh trong 24 giờ sau sinh (đơn giá); ...
2,3,2 tháng,2,2,3,DTP/5-in-1,1,0,tt10-2024,Bắt đầu từ đủ 2 tháng; thường dạng phối hợp có...
3,4,2 tháng,2,2,4,IPV,1,0,tt10-2024,Có thể nằm trong vaccine phối hợp tùy lịch
4,5,2 tháng,2,2,5,Hib,1,0,tt10-2024,Thường trong vaccine phối hợp
5,6,2 tháng,2,2,2,Viêm gan B,2,28,tt10-2024,Cách mũi sơ sinh tối thiểu 28 ngày
6,7,3-4 tháng,3,4,3,DTP/5-in-1,2,28,tt10-2024,Cách mũi 1 tối thiểu 28 ngày
7,8,3-4 tháng,3,4,4,IPV,2,28,tt10-2024,



--- Rules cho 14 tháng tuổi ---


,rule_id,age_group,age_from_months,age_to_months,vaccine_id,vaccine_short,dose_number,interval_days_min,source_doc,note
16,17,12-15 tháng,12,15,7,Thủy đậu,1,0,hdsd-varivax,Từ 12 tháng; HDSD VN 1 liều <13 tuổi — WHO khu...
17,18,12-15 tháng,12,15,8,PCV,1,0,who-pcv,Lịch 2+1 hoặc 3+1 tùy chương trình (dịch vụ tạ...
18,19,13-24 tháng,13,24,9,JE,2,14,tt10-2024,Cách mũi 1 khoảng 1-2 tuần (tối thiểu theo loạ...
34,35,12-23 tháng tiêm bù,12,23,6,Sởi,1,0,tt10-2024,Tiêm bù nếu chưa tiêm
35,36,Mọi lứa tuổi tiêm bù,0,1200,2,Viêm gan B,1,0,who-hepb,Phác đồ bù 0-1-6 tháng nếu chưa có
36,37,Mọi lứa tuổi tiêm bù,0,1200,3,Uốn ván,1,0,tt10-2024,Đủ 5 mũi trong đời



--- Expected doses map (14 tháng) ---
2 -> [('Viêm gan B', 1)]
3 -> [('Uốn ván', 1)]
6 -> [('Sởi', 1)]
7 -> [('Thủy đậu', 1)]
8 -> [('PCV', 1)]
9 -> [('JE', 2)]


In [7]:
# Thử với một mock user
u = mock_users[1]  # thiếu MMR / thủy đậu
print("User:", u["full_name"], "age_months=", u["age_months"])
exp = expected_doses_by_vaccine(u["age_months"])
have = {(r["vaccine_id"], r["dose_number"]) for r in u["records"]}
print("Đã có:", sorted(have))
missing = []
for vid, doses in exp.items():
    for d in doses:
        key = (vid, int(d["dose_number"]))
        if key not in have:
            missing.append((d["vaccine_short"], d["dose_number"], d["source_doc"]))
print("Có thể thiếu (theo rule hiện tại):", missing[:15])

User: Tran Thi B age_months= 18
Đã có: [(1, 1), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
Có thể thiếu (theo rule hiện tại): [('DTP nhắc', 4, 'tt10-2024'), ('Sởi', 1, 'tt10-2024'), ('MMR', 2, 'tt10-2024'), ('JE', 2, 'tt10-2024')]
